# High-z Example 17: Z1 vs SPARC Peak Velocity Comparison (EPS)

**EPS Research — Cross-Corpus**

How do Z1 peak rotation velocities compare to the SPARC z=0 sample?
Z1 tier-1 rotators span 62-252 km/s — overlapping with SPARC dwarfs
and intermediate spirals, but not massive spirals.

This cross-corpus comparison requires both Z1 and SPARC corpora.

**Corpus:** Flynn (2026), Zenodo DOI: 10.5281/zenodo.20369286  
**arXiv:** 2605.25339  
**Source:** Jones et al. (2021), MNRAS 507, 3540; Le Fevre et al. (2020)  
**Dependencies:** Python 3, numpy, matplotlib

In [ ]:
# ── Colab setup: auto-download corpus from Zenodo ─────────────
import os, sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    import urllib.request
    CORPORA = {
        'high_z_kinematic_corpus_Z1.json': 'https://zenodo.org/records/20369286/files/high_z_kinematic_corpus_Z1.json',
    }
    for filename, url in CORPORA.items():
        if not os.path.exists(filename):
            print(f"Downloading {filename}...")
            urllib.request.urlretrieve(url, filename)
            print(f"  ✓ {filename}")
        else:
            print(f"  Already present: {filename}")
    print("Ready.")
else:
    print("Running locally — corpus files loaded from working directory.")


In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt

# Load Z1
with open('high_z_kinematic_corpus_Z1.json') as f:
    z1_corpus = json.load(f)

rotators = [g for g in z1_corpus['galaxies']
            if g.get('is_rotator') and g.get('quality_tier')==1]
z1_vmax = [g['vrot_max_kms'] for g in rotators if g.get('vrot_max_kms')]
z1_names = [g['galaxy'] for g in rotators if g.get('vrot_max_kms')]

print(f"Z1 tier-1 Vmax range: {min(z1_vmax):.0f} -- {max(z1_vmax):.0f} km/s")
print(f"Z1 tier-1 median Vmax: {np.median(z1_vmax):.0f} km/s")

# SPARC reference from corpus v7 if available
try:
    import csv
    sparc_vmax = []
    with open('../hi/rotation_curve_corpus_v7_flat.csv') as f:
        for r in csv.DictReader(f):
            if r['survey']=='SPARC' and r.get('vrot_max_kms'):
                sparc_vmax.append(float(r['vrot_max_kms']))
    print(f"\nSPARC Vmax range: {min(sparc_vmax):.0f} -- {max(sparc_vmax):.0f} km/s")
    print(f"SPARC median Vmax: {np.median(sparc_vmax):.0f} km/s")
    have_sparc = True
except:
    print("\nSPARC corpus not found in ../hi/ — showing Z1 only")
    have_sparc = False

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
bins = range(0, 400, 20)

if have_sparc:
    ax.hist(sparc_vmax, bins=bins, alpha=0.5, color='#3498db',
            label=f'SPARC z=0 (N={len(sparc_vmax)})', edgecolor='white')
ax.hist(z1_vmax, bins=bins, alpha=0.8, color='#e74c3c',
        label=f'Z1 z~4-6 tier-1 (N={len(z1_vmax)})', edgecolor='white')

for v, name in zip(z1_vmax, z1_names):
    ax.axvline(v, color='#e74c3c', lw=0.5, alpha=0.4)

ax.set_xlabel('Peak rotation velocity (km/s)', fontsize=12)
ax.set_ylabel('N galaxies', fontsize=12)
ax.set_title('Z1 vs SPARC Peak Velocity Distribution\n'
             'EPS Research Cross-Corpus Comparison', fontsize=11)
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('hz17_z1_vs_sparc_vmax.png', dpi=150, bbox_inches='tight')
plt.show()